# Model Training and Evaluation
## Healthcare Provider Fraud Detection

This notebook focuses on building, training, and evaluating machine learning models for fraud detection.

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_score, recall_score, f1_score, accuracy_score,
    precision_recall_curve, auc
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# Configure display and plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
# Load processed data
X_train = pd.read_csv('X_train_processed.csv')
X_test = pd.read_csv('X_test_processed.csv')
y_train = pd.read_csv('y_train.csv').values.ravel()

print(f"Data loaded successfully!")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")

## 2. Train-Validation Split

In [ ]:
# Split into train and validation sets
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"\nTrain-Validation Split:")
print(f"  X_train_split: {X_train_split.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  y_train_split fraud distribution: {np.bincount(y_train_split)}")
print(f"  y_val fraud distribution: {np.bincount(y_val)}")

## 3. Model Training and Comparison

### 3.1 Logistic Regression

In [ ]:
# Logistic Regression
print("\n" + "="*60)
print("Training Logistic Regression...")
print("="*60)

lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_split, y_train_split)

# Predictions
lr_pred_val = lr_model.predict(X_val)
lr_proba_val = lr_model.predict_proba(X_val)[:, 1]

# Metrics
print(f"\nValidation Metrics:")
print(f"  Accuracy: {accuracy_score(y_val, lr_pred_val):.4f}")
print(f"  Precision: {precision_score(y_val, lr_pred_val):.4f}")
print(f"  Recall: {recall_score(y_val, lr_pred_val):.4f}")
print(f"  F1-Score: {f1_score(y_val, lr_pred_val):.4f}")
print(f"  ROC-AUC: {roc_auc_score(y_val, lr_proba_val):.4f}")

### 3.2 Random Forest

In [ ]:
# Random Forest
print("\n" + "="*60)
print("Training Random Forest...")
print("="*60)

rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=15, random_state=42, n_jobs=-1,
    class_weight='balanced'
)
rf_model.fit(X_train_split, y_train_split)

# Predictions
rf_pred_val = rf_model.predict(X_val)
rf_proba_val = rf_model.predict_proba(X_val)[:, 1]

# Metrics
print(f"\nValidation Metrics:")
print(f"  Accuracy: {accuracy_score(y_val, rf_pred_val):.4f}")
print(f"  Precision: {precision_score(y_val, rf_pred_val):.4f}")
print(f"  Recall: {recall_score(y_val, rf_pred_val):.4f}")
print(f"  F1-Score: {f1_score(y_val, rf_pred_val):.4f}")
print(f"  ROC-AUC: {roc_auc_score(y_val, rf_proba_val):.4f}")

### 3.3 Gradient Boosting

In [ ]:
# Gradient Boosting
print("\n" + "="*60)
print("Training Gradient Boosting...")
print("="*60)

gb_model = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42
)
gb_model.fit(X_train_split, y_train_split)

# Predictions
gb_pred_val = gb_model.predict(X_val)
gb_proba_val = gb_model.predict_proba(X_val)[:, 1]

# Metrics
print(f"\nValidation Metrics:")
print(f"  Accuracy: {accuracy_score(y_val, gb_pred_val):.4f}")
print(f"  Precision: {precision_score(y_val, gb_pred_val):.4f}")
print(f"  Recall: {recall_score(y_val, gb_pred_val):.4f}")
print(f"  F1-Score: {f1_score(y_val, gb_pred_val):.4f}")
print(f"  ROC-AUC: {roc_auc_score(y_val, gb_proba_val):.4f}")

### 3.4 XGBoost

In [ ]:
# XGBoost
print("\n" + "="*60)
print("Training XGBoost...")
print("="*60)

xgb_model = XGBClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42,
    scale_pos_weight=(y_train_split == 0).sum() / (y_train_split == 1).sum()
)
xgb_model.fit(X_train_split, y_train_split)

# Predictions
xgb_pred_val = xgb_model.predict(X_val)
xgb_proba_val = xgb_model.predict_proba(X_val)[:, 1]

# Metrics
print(f"\nValidation Metrics:")
print(f"  Accuracy: {accuracy_score(y_val, xgb_pred_val):.4f}")
print(f"  Precision: {precision_score(y_val, xgb_pred_val):.4f}")
print(f"  Recall: {recall_score(y_val, xgb_pred_val):.4f}")
print(f"  F1-Score: {f1_score(y_val, xgb_pred_val):.4f}")
print(f"  ROC-AUC: {roc_auc_score(y_val, xgb_proba_val):.4f}")

## 4. Model Comparison

In [ ]:
# Compare models
models_comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Gradient Boosting', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y_val, lr_pred_val),
        accuracy_score(y_val, rf_pred_val),
        accuracy_score(y_val, gb_pred_val),
        accuracy_score(y_val, xgb_pred_val)
    ],
    'Precision': [
        precision_score(y_val, lr_pred_val),
        precision_score(y_val, rf_pred_val),
        precision_score(y_val, gb_pred_val),
        precision_score(y_val, xgb_pred_val)
    ],
    'Recall': [
        recall_score(y_val, lr_pred_val),
        recall_score(y_val, rf_pred_val),
        recall_score(y_val, gb_pred_val),
        recall_score(y_val, xgb_pred_val)
    ],
    'F1-Score': [
        f1_score(y_val, lr_pred_val),
        f1_score(y_val, rf_pred_val),
        f1_score(y_val, gb_pred_val),
        f1_score(y_val, xgb_pred_val)
    ],
    'ROC-AUC': [
        roc_auc_score(y_val, lr_proba_val),
        roc_auc_score(y_val, rf_proba_val),
        roc_auc_score(y_val, gb_proba_val),
        roc_auc_score(y_val, xgb_proba_val)
    ]
})

print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)
print(models_comparison.to_string(index=False))

## 5. Best Model Evaluation

In [ ]:
# Select best model (XGBoost)
best_model = xgb_model
best_pred = xgb_pred_val
best_proba = xgb_proba_val
best_name = 'XGBoost'

print(f"\n{'='*60}")
print(f"BEST MODEL: {best_name}")
print(f"{'='*60}")

# Classification Report
print(f"\nClassification Report:")
print(classification_report(y_val, best_pred, target_names=['Non-Fraud', 'Fraud']))

### 5.1 Confusion Matrix

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_val, best_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Non-Fraud', 'Fraud'],
            yticklabels=['Non-Fraud', 'Fraud'])
plt.title(f"Confusion Matrix - {best_name}", fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print(f"\nConfusion Matrix:")
print(f"  True Negatives (TN): {cm[0, 0]}")
print(f"  False Positives (FP): {cm[0, 1]}")
print(f"  False Negatives (FN): {cm[1, 0]}")
print(f"  True Positives (TP): {cm[1, 1]}")

### 5.2 ROC Curve

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_val, best_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title(f'ROC Curve - {best_name}', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 5.3 Precision-Recall Curve

In [ ]:
# Precision-Recall Curve
precision, recall, thresholds = precision_recall_curve(y_val, best_proba)
pr_auc = auc(recall, precision)

plt.figure(figsize=(10, 8))
plt.plot(recall, precision, color='green', lw=2, label=f'PR curve (AUC = {pr_auc:.4f})')
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title(f'Precision-Recall Curve - {best_name}', fontsize=14, fontweight='bold')
plt.legend(loc="best", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot top 15 features
top_features = feature_importance.head(15)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=top_features, palette='viridis')
plt.title(f'Top 15 Important Features - {best_name}', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score', fontsize=12)
plt.tight_layout()
plt.show()

print(f"\nTop 15 Important Features:")
print(top_features.to_string(index=False))

## 7. Final Predictions on Test Set

In [ ]:
# Make predictions on test set
test_pred = best_model.predict(X_test)
test_proba = best_model.predict_proba(X_test)[:, 1]

# Create submission file
submission = pd.DataFrame({
    'BeneID': test_id['BeneID'].values,
    'PotentialFraud': test_pred,
    'FraudProbability': test_proba
})

submission.to_csv('fraud_detection_predictions.csv', index=False)

print(f"\nTest Predictions Summary:")
print(f"  Total predictions: {len(submission)}")
print(f"  Fraud cases predicted: {(submission['PotentialFraud'] == 1).sum()}")
print(f"  Non-fraud cases predicted: {(submission['PotentialFraud'] == 0).sum()}")
print(f"\n  Fraud probability distribution:")
print(submission['FraudProbability'].describe())
print(f"\nSubmission saved to: fraud_detection_predictions.csv")
print(submission.head(10))

## 8. Model Summary

In [ ]:
print(f"\n" + "="*80)
print(f"MODEL TRAINING COMPLETE - FINAL SUMMARY")
print(f"="*80)
print(f"\nBest Model: {best_name}")
print(f"\nValidation Performance:")
print(f"  Accuracy: {accuracy_score(y_val, best_pred):.4f}")
print(f"  Precision: {precision_score(y_val, best_pred):.4f}")
print(f"  Recall: {recall_score(y_val, best_pred):.4f}")
print(f"  F1-Score: {f1_score(y_val, best_pred):.4f}")
print(f"  ROC-AUC: {roc_auc_score(y_val, best_proba):.4f}")
print(f"\nOutputs:")
print(f"  - Trained model ready for deployment")
print(f"  - Fraud detection predictions on test set")
print(f"  - Feature importance analysis completed")